# Apart Club San Pedro — Real Estate Sales Agent
### LangGraph + Pinecone RAG + ReAct Pattern

**Run cells in order. Each cell = one future Python module.**

| Cell | What it tests | Future file |
|------|--------------|-------------|
| 1 | Imports + env | config |
| 2 | Pinecone connection | nodes/retrieve_context.py |
| 3 | RAG retrieval | nodes/retrieve_context.py |
| 4 | ReAct agent | nodes/react_agent.py |
| 5 | Lead classifier | nodes/classify_lead.py |
| 6 | Assemble output | nodes/assemble_output.py |
| 7 | LangGraph graph | graph.py |
| 8 | End-to-end tests | test_agent.py |

In [1]:
# ─────────────────────────────────────────────────────────────
# CELL 1 — Imports + Environment
# Future file: config / .env
# Run this first. If any import fails, pip install it.
# ─────────────────────────────────────────────────────────────

import os
import json
from dotenv import load_dotenv

# LangChain core
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

# Tavily
from langchain_community.tools.tavily_search import TavilySearchResults

# LangGraph
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
from typing import TypedDict, List

# Pinecone
from pinecone import Pinecone

# Load .env file
load_dotenv()

# ── Constants — must match your Pinecone index exactly ────────
PINECONE_INDEX_NAME = "n8n"
EMBEDDING_MODEL     = "text-embedding-3-large"
LLM_MODEL           = "gpt-4o-mini"

# ── Verify keys are loaded ────────────────────────────────────
required_keys = ["OPENAI_API_KEY", "PINECONE_API_KEY", "TAVILY_API_KEY"]
missing = [k for k in required_keys if not os.getenv(k)]

if missing:
    print(f"❌ Missing env vars: {missing}")
    print("Create a .env file with these keys before continuing.")
else:
    print("✅ All API keys loaded")
    print(f"   Pinecone index : {PINECONE_INDEX_NAME}")
    print(f"   Embedding model: {EMBEDDING_MODEL}")
    print(f"   LLM model      : {LLM_MODEL}")

✅ All API keys loaded
   Pinecone index : n8n
   Embedding model: text-embedding-3-large
   LLM model      : gpt-4o-mini


In [2]:
# ─────────────────────────────────────────────────────────────
# CELL 2 — Pinecone Connection Test
# Just pings the index and prints stats.
# Expected output: index name, vector count > 0
# ─────────────────────────────────────────────────────────────

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index(PINECONE_INDEX_NAME)

stats = index.describe_index_stats()
print(f"✅ Connected to Pinecone index: '{PINECONE_INDEX_NAME}'")
print(f"   Total vectors : {stats['total_vector_count']}")
print(f"   Dimension     : {stats['dimension']}")

if stats['total_vector_count'] == 0:
    print("⚠️  Index is empty — docs may not have been ingested yet")
else:
    print("   Documents are loaded and ready ✅")

✅ Connected to Pinecone index: 'n8n'
   Total vectors : 909
   Dimension     : 1024
   Documents are loaded and ready ✅


In [5]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — RAG Retrieval Test
# Future file: nodes/retrieve_context.py
#
# Sends a test query and prints the top 4 chunks.
# Check: are the returned chunks actually about prices / lots?
# If chunks look wrong, the namespace may differ — check below.
# ─────────────────────────────────────────────────────────────

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    dimensions=1024  # must match n8n ingestion
)

vectorstore = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings,
    # namespace=""  # uncomment and set if n8n used a specific namespace
)

TEST_QUERY = "¿Cuánto cuesta una parcela y cuáles son los planes de pago?"

docs = vectorstore.similarity_search(TEST_QUERY, k=4)

print(f"Query: {TEST_QUERY}")
print(f"Chunks retrieved: {len(docs)}\n")

for i, doc in enumerate(docs):
    print(f"── Chunk {i+1} ──────────────────────────────")
    print(f"Source  : {doc.metadata.get('source', 'unknown')}")
    print(f"Content : {doc.page_content[:300]}")
    print()

# ── Helper function (will become retrieve_context.py) ─────────
def retrieve_context(user_message: str, k: int = 4) -> str:
    """Query Pinecone and return formatted context string."""
    docs = vectorstore.similarity_search(user_message, k=k)
    parts = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "unknown")
        parts.append(f"[Chunk {i+1} — {source}]\n{doc.page_content}")
    return "\n\n".join(parts)

Query: ¿Cuánto cuesta una parcela y cuáles son los planes de pago?
Chunks retrieved: 4

── Chunk 1 ──────────────────────────────
Source  : blob
Content : PROPUESTA DE VENTA 3:  Venta de LOTES 
 
o Venta de lotes dentro de cada parcela. 
 
o Valor de los lotes a U$S120 el m2. 
 
Forma de pago: Reserva = 50% 
                             Financiación = 15 cuotas 
 
 
COMISIÓN POR VENTA en la PROPUESTA 3 = 10% 
                             
 
 
       V

── Chunk 2 ──────────────────────────────
Source  : blob
Content : PROPUESTA DE VENTA 3:  Venta de LOTES 
 
o Venta de lotes dentro de cada parcela. 
 
o Valor de los lotes a U$S120 el m2. 
 
Forma de pago: Reserva = 50% 
                             Financiación = 15 cuotas 
 
 
COMISIÓN POR VENTA en la PROPUESTA 3 = 10% 
                             
 
 
       V

── Chunk 3 ──────────────────────────────
Source  : blob
Content : PROPUESTA DE VENTA 3:  Venta de LOTES 
 
o Venta de lotes dentro de cada parcela. 
 
o Valor de los lotes a

In [8]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

tavily_tool = TavilySearchResults(max_results=3)
llm = ChatOpenAI(model=LLM_MODEL, temperature=0.2)
tools = [tavily_tool]

SYSTEM_PROMPT = """You are a professional real estate sales assistant for Apart Club San Pedro,
a residential nautical club development in San Pedro, Buenos Aires, Argentina.
Answer in the same language as the lead's question (Spanish or English).
Be specific with numbers — prices, lot sizes, payment plans — when available in the context.
Use the tavily_search tool ONLY when the lead asks how prices compare to nearby developments
or asks about the broader real estate market. Do NOT use it for questions answerable from context.
Always end with a clear, professional Final Answer."""

react_agent_runnable = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

TEST_MESSAGE = "¿Cuánto cuesta una parcela y cuáles son los planes de pago?"
context = retrieve_context(TEST_MESSAGE)

full_input = f"PROPERTY CONTEXT:\n{context}\n\n---\nLead question: {TEST_MESSAGE}"

result = react_agent_runnable.invoke({"messages": [HumanMessage(content=full_input)]})
final_answer = result["messages"][-1].content

print("=" * 60)
print("FINAL ANSWER:")
print(final_answer)

def run_react_agent(user_message: str, context: str) -> str:
    full_input = f"PROPERTY CONTEXT:\n{context}\n\n---\nLead question: {user_message}"
    result = react_agent_runnable.invoke({"messages": [HumanMessage(content=full_input)]})
    return result["messages"][-1].content

C:\Users\pbiai\AppData\Local\Temp\ipykernel_21716\2717575091.py:6: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=3)
C:\Users\pbiai\AppData\Local\Temp\ipykernel_21716\2717575091.py:18: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  react_agent_runnable = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)


FINAL ANSWER:
El costo de cada parcela es de U$S 45 por m². La forma de pago incluye una reserva del 50% y financiación en 10 cuotas. 

Si necesitas más detalles sobre las superficies de las parcelas o cualquier otra información, no dudes en preguntar.

Final Answer: Cada parcela cuesta U$S 45 por m², con un plan de pago que incluye una reserva del 50% y financiación en 10 cuotas.


In [13]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — Lead Classifier Test
# Future file: nodes/classify_lead.py
#
# Scores the lead as HOT / WARM / COLD.
# Uses a structured LLM call — no tools needed.
# Tests all 3 score levels.
# ─────────────────────────────────────────────────────────────

CLASSIFICATION_SYSTEM = """
You are a lead scoring assistant for a real estate development in Argentina.
Given the lead's message, classify them as:

HOT  — Mentions a specific budget, asks about a specific lot number,
        asks about payment plans with intent to buy, mentions investment ROI,
        or signals they are ready to proceed.
WARM — General interest, asking about prices or availability,
        asking about building regulations or construction rules,
        curious but no urgency or budget mentioned.
COLD — Just browsing, very vague, hypothetical, or no real intent signals.

Respond ONLY with valid JSON, no extra text:
{"score": "HOT", "reason": "one sentence explanation"}
"""

classifier_llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

def classify_lead(user_message: str) -> dict:
    """Classify lead and return {score, reason} dict."""
    response = classifier_llm.invoke([
        SystemMessage(content=CLASSIFICATION_SYSTEM),
        HumanMessage(content=f"Lead message: {user_message}"),
    ])
    try:
        return json.loads(response.content)
    except json.JSONDecodeError:
        return {"score": "COLD", "reason": "parse error — defaulting to COLD"}

# ── Test all 3 levels ─────────────────────────────────────────
test_messages = [
    ("HOT",  "Tengo U$S 50.000 disponibles. ¿Qué parcelas entran en mi presupuesto y cómo son los planes de pago?"),
    ("WARM", "Me interesa saber cuánto cuestan las parcelas en el Club Náutico."),
    ("COLD", "¿Tienen algo en la costa? Solo estoy mirando opciones."),
]

print("Lead Classification Tests\n" + "="*50)
all_passed = True

for expected, message in test_messages:
    result = classify_lead(message)
    match = result['score'] == expected
    if not match:
        all_passed = False
    print(f"{'✅' if match else '❌'} Expected: {expected} | Got: {result['score']}")
    print(f"   Message : {message[:70]}...")
    print(f"   Reason  : {result['reason']}\n")

print("All classification tests passed ✅" if all_passed else "Some tests failed — tune the prompt ⚠️")

Lead Classification Tests
✅ Expected: HOT | Got: HOT
   Message : Tengo U$S 50.000 disponibles. ¿Qué parcelas entran en mi presupuesto y...
   Reason  : Mentions a specific budget and asks about payment plans with intent to buy.

✅ Expected: WARM | Got: WARM
   Message : Me interesa saber cuánto cuestan las parcelas en el Club Náutico....
   Reason  : The lead is asking about prices but does not mention a specific budget or intent to buy.

✅ Expected: COLD | Got: COLD
   Message : ¿Tienen algo en la costa? Solo estoy mirando opciones....
   Reason  : The lead is just browsing and has no real intent signals.

All classification tests passed ✅


In [10]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — Assemble Output Test
# Future file: nodes/assemble_output.py
#
# Packages everything into the JSON blob n8n will read.
# The flags send_telegram / send_email drive n8n routing.
# ─────────────────────────────────────────────────────────────

def assemble_output(
    user_message: str,
    agent_response: str,
    lead_score: str,
    actions_taken: list
) -> dict:
    """Build final JSON output for n8n."""
    return {
        "answer":        agent_response,
        "score":         lead_score,
        "original_message": user_message,
        "actions_taken": actions_taken,
        # n8n reads these flags to decide which nodes to trigger
        "send_telegram": lead_score == "HOT",
        "send_email":    lead_score in ("HOT", "WARM"),
        "log_to_sheets": True,
    }

# ── Test it ───────────────────────────────────────────────────
sample_output = assemble_output(
    user_message   = "Tengo U$S 50.000, ¿qué parcelas puedo comprar?",
    agent_response = "Con U$S 50.000 puede adquirir la parcela 12 de 544m2...",
    lead_score     = "HOT",
    actions_taken  = ["rag_retrieval", "react_agent", "classified_as_HOT"]
)

print("Assembled output JSON:")
print(json.dumps(sample_output, ensure_ascii=False, indent=2))

# Verify flags
assert sample_output["send_telegram"] == True,  "HOT should trigger Telegram"
assert sample_output["send_email"]    == True,  "HOT should trigger email"
assert sample_output["log_to_sheets"] == True,  "Always log"
print("\n✅ Output assembly working correctly")

Assembled output JSON:
{
  "answer": "Con U$S 50.000 puede adquirir la parcela 12 de 544m2...",
  "score": "HOT",
  "original_message": "Tengo U$S 50.000, ¿qué parcelas puedo comprar?",
  "actions_taken": [
    "rag_retrieval",
    "react_agent",
    "classified_as_HOT"
  ],
  "send_telegram": true,
  "send_email": true,
  "log_to_sheets": true
}

✅ Output assembly working correctly


In [11]:
# ─────────────────────────────────────────────────────────────
# CELL 7 — Full LangGraph Graph
# Future file: graph.py
#
# Wires all nodes into the state machine.
# State flows: input → RAG → ReAct → classify → route → output
# ─────────────────────────────────────────────────────────────

# ── State definition ──────────────────────────────────────────
class AgentState(TypedDict):
    user_message:      str
    retrieved_context: str
    agent_response:    str
    lead_score:        str
    score_reason:      str
    actions_taken:     List[str]
    final_output:      dict

# ── Node functions ────────────────────────────────────────────
def node_retrieve_context(state: AgentState) -> AgentState:
    context = retrieve_context(state["user_message"])  # from Cell 3
    state["retrieved_context"] = context
    state["actions_taken"].append("rag_retrieval")
    return state

def node_react_agent(state: AgentState) -> AgentState:
    response = run_react_agent(                        # from Cell 4
        state["user_message"],
        state["retrieved_context"]
    )
    state["agent_response"] = response
    state["actions_taken"].append("react_agent")
    return state

def node_classify_lead(state: AgentState) -> AgentState:
    result = classify_lead(state["user_message"])      # from Cell 5
    state["lead_score"]   = result["score"]
    state["score_reason"] = result["reason"]
    state["actions_taken"].append(f"classified_as_{result['score']}")
    return state

def node_assemble_output(state: AgentState) -> AgentState:
    state["final_output"] = assemble_output(           # from Cell 6
        state["user_message"],
        state["agent_response"],
        state["lead_score"],
        state["actions_taken"]
    )
    return state

# ── Conditional routing ───────────────────────────────────────
def route_by_score(state: AgentState) -> str:
    score = state.get("lead_score", "COLD")
    if score == "HOT":
        return "assemble_hot"
    elif score == "WARM":
        return "assemble_warm"
    return "assemble_cold"

# ── Build graph ───────────────────────────────────────────────
def build_graph():
    graph = StateGraph(AgentState)

    graph.add_node("retrieve_context", node_retrieve_context)
    graph.add_node("react_agent",      node_react_agent)
    graph.add_node("classify_lead",    node_classify_lead)
    # Three named targets — same function, different labels for clarity
    # Day 3: split these to fire Telegram/Gmail directly from Python
    graph.add_node("assemble_hot",  node_assemble_output)
    graph.add_node("assemble_warm", node_assemble_output)
    graph.add_node("assemble_cold", node_assemble_output)

    graph.set_entry_point("retrieve_context")
    graph.add_edge("retrieve_context", "react_agent")
    graph.add_edge("react_agent",      "classify_lead")

    graph.add_conditional_edges(
        "classify_lead",
        route_by_score,
        {
            "assemble_hot":  "assemble_hot",
            "assemble_warm": "assemble_warm",
            "assemble_cold": "assemble_cold",
        }
    )

    graph.add_edge("assemble_hot",  END)
    graph.add_edge("assemble_warm", END)
    graph.add_edge("assemble_cold", END)

    return graph.compile()

graph = build_graph()
print("✅ LangGraph compiled successfully")
print("   Nodes: retrieve_context → react_agent → classify_lead → route → assemble")

✅ LangGraph compiled successfully
   Nodes: retrieve_context → react_agent → classify_lead → route → assemble


In [14]:
# ─────────────────────────────────────────────────────────────
# CELL 8 — End-to-End Test Suite
# Future file: test_agent.py
#
# Runs 3 scenarios through the full graph.
# Day 2 milestone = all 3 pass.
# ─────────────────────────────────────────────────────────────

TEST_CASES = [
    {
        "id": 1,
        "description": "Basic price question",
        "message": "¿Cuánto cuesta una parcela en el Club Náutico?",
        "expect_score": "WARM",
        "expect_send_telegram": False,
        "expect_send_email": True,
    },
    {
        "id": 2,
        "description": "Specific budget + payment plan (HOT)",
        "message": "Tengo un presupuesto de U$S 50.000. ¿Qué parcelas puedo comprar y cuáles son los planes de pago?",
        "expect_score": "HOT",
        "expect_send_telegram": True,
        "expect_send_email": True,
    },
    {
        "id": 3,
        "description": "Building regulations question",
        "message": "¿Puedo construir un edificio de departamentos en mi parcela?",
        "expect_score": "WARM",
        "expect_send_telegram": False,
        "expect_send_email": True,
    },
]

def run_test(case: dict) -> bool:
    print(f"\nTEST {case['id']}: {case['description']}")
    print(f"Input : {case['message']}")
    print("-" * 55)

    initial_state: AgentState = {
        "user_message":      case["message"],
        "retrieved_context": "",
        "agent_response":    "",
        "lead_score":        "",
        "score_reason":      "",
        "actions_taken":     [],
        "final_output":      {},
    }

    result = graph.invoke(initial_state)
    output = result["final_output"]

    score_ok     = output["score"] == case["expect_score"]
    telegram_ok  = output["send_telegram"] == case["expect_send_telegram"]
    email_ok     = output["send_email"] == case["expect_send_email"]
    passed       = score_ok and telegram_ok and email_ok

    print(f"Score    : {output['score']:4s} (expected {case['expect_score']:4s}) {'✅' if score_ok else '❌'}")
    print(f"Telegram : {str(output['send_telegram']):5s} (expected {str(case['expect_send_telegram']):5s}) {'✅' if telegram_ok else '❌'}")
    print(f"Email    : {str(output['send_email']):5s} (expected {str(case['expect_send_email']):5s}) {'✅' if email_ok else '❌'}")
    print(f"Answer   : {output['answer'][:150]}...")
    print(f"Actions  : {output['actions_taken']}")
    return passed

# ── Run all tests ─────────────────────────────────────────────
print("=" * 55)
print("END-TO-END TEST SUITE — Apart Club San Pedro Agent")
print("=" * 55)

results = [run_test(case) for case in TEST_CASES]
passed  = sum(results)

print(f"\n{'='*55}")
print(f"RESULTS: {passed}/{len(TEST_CASES)} tests passed")

if passed == len(TEST_CASES):
    print("\n🎉 Day 2 milestone complete — ready to convert to .py files")
else:
    print("\n⚠️  Some tests failed — debug the failing cells above first")

END-TO-END TEST SUITE — Apart Club San Pedro Agent

TEST 1: Basic price question
Input : ¿Cuánto cuesta una parcela en el Club Náutico?
-------------------------------------------------------
Score    : WARM (expected WARM) ✅
Telegram : False (expected False) ✅
Email    : True  (expected True ) ✅
Answer   : El costo de una parcela en el Club Náutico es de U$ 135 por metro cuadrado. Dado que se están ofreciendo 46 parcelas con una superficie total de 30.50...
Actions  : ['rag_retrieval', 'react_agent', 'classified_as_WARM']

TEST 2: Specific budget + payment plan (HOT)
Input : Tengo un presupuesto de U$S 50.000. ¿Qué parcelas puedo comprar y cuáles son los planes de pago?
-------------------------------------------------------
Score    : HOT  (expected HOT ) ✅
Telegram : True  (expected True ) ✅
Email    : True  (expected True ) ✅
Answer   : Con un presupuesto de U$S 50.000, puedes realizar una reserva para la compra de parcelas en Apart Club San Pedro. Aquí están las opciones disponibl